# Practice 3: Customer Clustering - Bước 5: Xử lý đặc trưng (Feature Engineering)

---

## 1. Import các thư viện và tải dữ liệu đã làm sạch

Tải tệp `Train_cleaned.csv` từ bước làm sạch dữ liệu.

In [55]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Cấu hình vẽ đồ thị
%matplotlib inline
sns.set_theme(style="whitegrid")

# Load dữ liệu sạch
df = pd.read_csv("../data/processed/Train_cleaned.csv")
print(f"Kích thước dữ liệu sạch: {df.shape}")
df.head(10)

Kích thước dữ liệu sạch: (7422, 10)


,ID,Gender,Ever_Married,Age,Graduated,Profession,Work_Experience,Spending_Score,Family_Size,Var_1
0,462809,Male,No,22,No,Healthcare,1.0,Low,4.0,Cat_4
1,462643,Female,Yes,38,Yes,Engineer,1.0,Average,3.0,Cat_4
2,466315,Female,Yes,67,Yes,Engineer,1.0,Low,1.0,Cat_6
3,461735,Male,Yes,67,Yes,Lawyer,0.0,High,2.0,Cat_6
4,462669,Female,Yes,40,Yes,Entertainment,1.0,High,6.0,Cat_6
5,461319,Male,Yes,56,No,Artist,0.0,Average,2.0,Cat_6
6,460156,Male,No,32,Yes,Healthcare,1.0,Low,3.0,Cat_6
7,464347,Female,No,33,Yes,Healthcare,1.0,Low,3.0,Cat_6
8,465015,Female,Yes,61,Yes,Engineer,0.0,Low,3.0,Cat_7
9,465176,Female,Yes,55,Yes,Artist,1.0,Average,4.0,Cat_6


## 2. Phân tách tập dữ liệu

Chúng ta tách cột định danh `ID` ra khỏi tập đặc trưng huấn luyện $X$.

In [46]:
customer_ids = df['ID'].values

X = df.drop(columns=['ID'])
print('Các cột đặc trưng đầu vào X:')
print(list(X.columns))

Các cột đặc trưng đầu vào X:
['Gender', 'Ever_Married', 'Age', 'Graduated', 'Profession', 'Work_Experience', 'Spending_Score', 'Family_Size', 'Var_1']


## 4. Tự viết bộ chuẩn hóa dữ liệu số học (Custom Scalers từ Scratch)

Chúng ta sẽ xây dựng lớp `CustomMinMaxScaler` từ scratch để đưa tất cả các đặc trưng số về khoảng $[0, 1]$, cân bằng thang đo với các biến One-Hot.

In [47]:
class CustomMinMaxScaler:
    def __init__(self, feature_range=(0, 1)):
        self.feature_range = feature_range
        self.min_ = None
        self.max_ = None
        self.range_ = None
        
    def fit(self, X):
        X_arr = np.array(X)
        self.min_ = np.min(X_arr, axis=0)
        self.max_ = np.max(X_arr, axis=0)
        self.range_ = self.max_ - self.min_
        # Tránh lỗi chia cho 0 nếu min == max
        self.range_ = np.where(self.range_ == 0, 1e-8, self.range_)
        return self
        
    def transform(self, X):
        X_arr = np.array(X)
        X_std = (X_arr - self.min_) / self.range_
        return X_std * (self.feature_range[1] - self.feature_range[0]) + self.feature_range[0]
        
    def fit_transform(self, X):
        self.fit(X)
        return self.transform(X)

## 5. Tự viết bộ mã hóa biến phân loại từ Scratch

Mã hóa `Spending_Score` dạng Ordinal và One-Hot cho các nominal features còn lại.

In [48]:
# 5.1. Ordinal Encoding cho Spending_Score (Đưa về khoảng [0, 1] để đồng nhất khoảng cách)
spending_mapping = {'Low': 0.0, 'Average': 0.5, 'High': 1.0}
X['Spending_Score'] = X['Spending_Score'].map(spending_mapping)

# 5.2. Custom One-Hot Encoder
class CustomOneHotEncoder:
    def __init__(self):
        self.categories_ = {}
        
    def fit(self, df, columns):
        self.columns = columns
        for col in columns:
            self.categories_[col] = sorted(list(df[col].unique()))
        return self
        
    def transform(self, df):
        df_out = df.copy()
        for col in self.columns:
            cats = self.categories_[col]
            for cat in cats:
                new_col_name = f'{col}_{cat}'
                df_out[new_col_name] = (df_out[col] == cat).astype(int)
            df_out = df_out.drop(columns=[col])
        return df_out
        
    def fit_transform(self, df, columns):
        self.fit(df, columns)
        return self.transform(df)

nominal_cols = ['Gender', 'Ever_Married', 'Graduated', 'Profession', 'Var_1']
encoder = CustomOneHotEncoder()
X_encoded = encoder.fit_transform(X, nominal_cols)
print(f'Hình dạng ma trận sau One-Hot: {X_encoded.shape}')

Hình dạng ma trận sau One-Hot: (7422, 26)


## 6. Tự viết lớp Giảm chiều dữ liệu PCA từ Scratch (Custom PCA)

Chúng ta tự xây dựng thuật toán PCA dựa trên đại số tuyến tính để khảo sát mức độ cô đọng thông tin của dữ liệu thưa.

In [49]:
class CustomPCA:
    def __init__(self, n_components):
        self.n_components = n_components
        self.components_ = None
        self.mean_ = None
        self.eigenvalues_ = None
        
    def fit(self, X):
        X_arr = np.array(X, dtype=float)
        # 1. Định tâm dữ liệu
        self.mean_ = np.mean(X_arr, axis=0)
        X_centered = X_arr - self.mean_
        
        # 2. Tính ma trận hiệp phương sai tổng thể (population covariance với ddof=0)
        cov_matrix = np.cov(X_centered, rowvar=False, ddof=0)
        
        # 3. Tìm trị riêng và vectơ riêng
        eigenvalues, eigenvectors = np.linalg.eigh(cov_matrix)
        
        # 4. Sắp xếp giảm dần
        sorted_indices = np.argsort(eigenvalues)[::-1]
        self.eigenvalues_ = eigenvalues[sorted_indices]
        sorted_eigenvectors = eigenvectors[:, sorted_indices]
        
        # Chọn n_components đầu tiên
        self.components_ = sorted_eigenvectors[:, :self.n_components]
        return self
        
    def transform(self, X):
        X_arr = np.array(X, dtype=float)
        X_centered = X_arr - self.mean_
        return np.dot(X_centered, self.components_)
        
    def fit_transform(self, X):
        self.fit(X)
        return self.transform(X)
        
    def explained_variance_ratio(self):
        total_var = np.sum(self.eigenvalues_)
        return self.eigenvalues_[:self.n_components] / total_var

## 7. Xây dựng quy trình xử lý đặc trưng và Khảo sát PCA

Chúng ta thực hiện chuẩn hóa khoảng $[0, 1]$ cho các cột số để cân bằng tuyệt đối với các thuộc tính One-Hot (MinMax Scaling).

In [50]:
# Chuẩn hóa MinMax cho các biến số học và ordinal (đồng bộ hóa dải dữ liệu)
numerical_cols = ['Age', 'Work_Experience', 'Family_Size', 'Spending_Score']
scaler_minmax = CustomMinMaxScaler()

X_scaled = X_encoded.copy()
X_scaled[numerical_cols] = scaler_minmax.fit_transform(X_encoded[numerical_cols])

print('Mẫu dữ liệu sau khi chuẩn hóa MinMax [0, 1] đồng bộ:')
print(X_scaled[numerical_cols].head(5))

Mẫu dữ liệu sau khi chuẩn hóa MinMax [0, 1] đồng bộ:
        Age  Work_Experience  Family_Size  Spending_Score
0  0.056338         0.071429        0.375             0.0
1  0.281690         0.071429        0.250             0.5
2  0.690141         0.071429        0.000             0.0
3  0.690141         0.000000        0.125             1.0
4  0.309859         0.071429        0.625             1.0


In [59]:
# Khảo sát giảm chiều PCA để xem lượng thông tin giữ lại
pca = CustomPCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

print(f'Tỷ lệ phương sai giải thích được của 2 thành phần: {pca.explained_variance_ratio()}')
print(f'Tổng lượng thông tin giữ lại ở không gian 2D: {np.sum(pca.explained_variance_ratio())*100:.2f}%')


Tỷ lệ phương sai giải thích được của 2 thành phần: [0.23364897 0.17484731]
Tổng lượng thông tin giữ lại ở không gian 2D: 40.85%


In [52]:
# Gán toàn bộ dữ liệu đã tiền xử lý để phục vụ huấn luyện phân cụm
X_train = X_scaled

# Load nhãn mục tiêu tương ứng đã được cô lập từ Bước 3
y_train = pd.read_csv('../data/processed_features/y_train.csv')['Segmentation'].values

print(f'Kích thước dữ liệu đặc trưng huấn luyện đầy đủ: {X_train.shape}')
print(f'Kích thước nhãn mục tiêu tương ứng:          {y_train.shape}')

Kích thước dữ liệu đặc trưng huấn luyện đầy đủ: (7422, 26)
Kích thước nhãn mục tiêu tương ứng:          (7422,)


## 9. Trực quan hóa dữ liệu đặc trưng huấn luyện trước khi lưu trữ

Chúng ta in ra 20 dòng dữ liệu đặc trưng huấn luyện đầu tiên (`X_train.head(20)`) để trực quan hóa cấu trúc dữ liệu sau khi kết thúc xử lý đặc trưng.

In [53]:
print('20 dòng dữ liệu đặc trưng huấn luyện đầu tiên (X_train):')
display(X_train.head(20))

20 dòng dữ liệu đặc trưng huấn luyện đầu tiên (X_train):


,Age,Work_Experience,Spending_Score,Family_Size,Gender_Female,Gender_Male,Ever_Married_No,Ever_Married_Yes,Graduated_No,Graduated_Yes,...,Profession_Homemaker,Profession_Lawyer,Profession_Marketing,Var_1_Cat_1,Var_1_Cat_2,Var_1_Cat_3,Var_1_Cat_4,Var_1_Cat_5,Var_1_Cat_6,Var_1_Cat_7
0,0.056338,0.071429,0.0,0.375,0,1,1,0,1,0,...,0,0,0,0,0,0,1,0,0,0
1,0.281690,0.071429,0.5,0.250,1,0,0,1,0,1,...,0,0,0,0,0,0,1,0,0,0
2,0.690141,0.071429,0.0,0.000,1,0,0,1,0,1,...,0,0,0,0,0,0,0,0,1,0
3,0.690141,0.000000,1.0,0.125,0,1,0,1,0,1,...,0,1,0,0,0,0,0,0,1,0
4,0.309859,0.071429,1.0,0.625,1,0,0,1,0,1,...,0,0,0,0,0,0,0,0,1,0
5,0.535211,0.000000,0.5,0.125,0,1,0,1,1,0,...,0,0,0,0,0,0,0,0,1,0
6,0.197183,0.071429,0.0,0.250,0,1,1,0,0,1,...,0,0,0,0,0,0,0,0,1,0
7,0.211268,0.071429,0.0,0.250,1,0,1,0,0,1,...,0,0,0,0,0,0,0,0,1,0
8,0.605634,0.000000,0.0,0.250,1,0,0,1,0,1,...,0,0,0,0,0,0,0,0,0,1
9,0.521127,0.071429,0.5,0.375,1,0,0,1,0,1,...,0,0,0,0,0,0,0,0,1,0


## 10. Lưu trữ dữ liệu xử lý đặc trưng đầy đủ

Chúng ta sẽ xuất bộ đặc trưng đầy đủ 22 chiều và nhãn tương ứng để làm đầu vào cho các thuật toán phân cụm.

In [54]:
import os
os.makedirs('../data/processed_features/', exist_ok=True)

# Lưu toàn bộ tập dữ liệu sạch để phân cụm trực tiếp
X_train.to_csv('../data/processed_features/X_train.csv', index=False)

# Lưu toàn bộ nhãn thực tế tương ứng
pd.DataFrame(y_train, columns=['Segmentation']).to_csv('../data/processed_features/y_train.csv', index=False)

print('Đã lưu toàn bộ dữ liệu đặc trưng đầy đủ chiều thành công!')

Đã lưu toàn bộ dữ liệu đặc trưng đầy đủ chiều thành công!
